## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [38]:
import os
import asyncio
from typing import Any
from dotenv import load_dotenv

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, trace, function_tool

from tavily import TavilyClient
from pydantic import BaseModel, Field
# OpenTelemetry + Phoenix tracing (matching lab1test.ipynb pattern)
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry import trace as otel_trace
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from typing import Dict


In [39]:
# ============================================================================
# 2. ENVIRONMENT SETUP
# ============================================================================

load_dotenv(override=True)

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
PHOENIX_OTLP_GRPC = os.environ.get("PHOENIX_OTLP_GRPC", "192.168.0.111:30317")
PHOENIX_UI = os.environ.get("PHOENIX_UI", "http://192.168.0.111:30606")

YAHOO_EMAIL = os.environ.get("YAHOO_EMAIL")
YAHOO_APP_PASSWORD = os.environ.get("YAHOO_APP_PASSWORD") 


print(f"✓ GROQ_API_KEY loaded")
print(f"✓ TAVILY_API_KEY loaded")


# Configure tracing — sends spans to Phoenix over gRPC (plain, no TLS)

provider = TracerProvider()
provider.add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint=PHOENIX_OTLP_GRPC, insecure=True))
)
otel_trace.set_tracer_provider(provider)
OpenAIAgentsInstrumentor().instrument(tracer_provider=provider)

print("Trace setup done")

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


✓ GROQ_API_KEY loaded
✓ TAVILY_API_KEY loaded
Trace setup done


In [40]:
tavily = TavilyClient(api_key=TAVILY_API_KEY)

@function_tool
def tavily_search(query: str) -> str:
    """Search the web using Tavily and return results as a string."""
    results = tavily.search(query=query, max_results=5)
    return "\n\n".join(
        f"{r['title']}\n{r['url']}\n{r['content']}"
        for r in results.get("results", [])
    )

tools = [tavily_search]

groq_client = AsyncOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

# vllm = AsyncOpenAI(
#     api_key=os.getenv("VLLM_API_KEY", "not-needed"),
#     base_url=os.getenv("VLLM_BASE_URL", "http://192.168.0.80:31080/v1")
#  )

model=OpenAIChatCompletionsModel(openai_client=groq_client, model="meta-llama/llama-4-scout-17b-16e-instruct")


In [41]:
# INSTRUCTIONS = """You are a research assistant. Given a search term, you search the web for that term and \
# produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 \
# words. Capture the main points. Write succinctly, no need for complete sentences or good grammar. \
# This will be consumed by someone synthesizing a report, so capture the essence and ignore fluff. \
# Do not include any additional commentary other than the summary itself. Write in flowing prose, not lists. Synthesize across sources rather than 
# listing frameworks one by one. Focus on patterns and key distinctions.

# CRITICAL RULES:
# 1. You MUST call tavily_search BEFORE writing anything
# 2. Base your summary ONLY on the tavily_search results
# 3. Output the summary directly — no preamble like 'Search results:' or 'Here is a summary:'
# """

INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term using the tool tavily_search and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself. Its important to Include links in the summary"


search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    model=model,
    tools=tools,
)


In [25]:
# Run search agent once (response should be: Search results: {raw_tavily_results})
message = "Latest AI Agent frameworks in 2026"
result = await Runner.run(search_agent, message)
print(result.final_output)

The current top AI agent frameworks in 2026 include LangGraph, CrewAI, Microsoft AutoGen, OpenAgents, and MetaGPT. LangGraph is a graph-native, stateful agent architecture built for persistent memory and multi-agent systems. CrewAI and AutoGen are also highly regarded frameworks. OpenAI Agents SDK and Google ADK are other notable frameworks. When selecting a framework, consider factors such as architecture, memory, and multi-agent capabilities. Companies are shifting towards agentic AI, where autonomous agents plan, reason, and coordinate. 

Links: 
- https://www.salesforce.com/agentforce/ai-agents/ai-agent-frameworks/
- https://www.intuz.com/blog/top-5-ai-agent-frameworks-2025
- https://medium.com/javarevisited/i-tried-20-ai-frameworks-here-are-my-top-10-recommendations-for-2026-927168fed61c
- https://rhesis.ai/post/picking-agentic-framework-2026 
- https://www.linkedin.com/posts/krishna-reddy-780a7286_the-4-agent-frameworks-that-will-define-ai-activity-7399658658479050752-klhL


In [26]:
with trace("Search Agent"):
    message = "Latest AI Agent frameworks in 2026"
    result = await Runner.run(search_agent, message)
    print(result.final_output)

Top AI agent frameworks in 2026 include Lindy, Mastra, LangChain, CrewAI, OpenAI Responses API, AutoGen, LlamaIndex, LangGraph, Haystack Agents, FastAgency, and Agentic AI Frameworks. Popular frameworks also include CrewAI, LangGraph, AutoGen, LlamaIndex, AutoAgent, DSPy, Haystack, and Microsoft Semantic Kernel. These frameworks offer varying capabilities and are used for different applications. Some of the key players in the market include LangGraph, CrewAI, AutoGen, and Semantic Kernel, which set the standard for multi-agent frameworks. 

LangGraph provides a graph-native, stateful agent architecture built for persistent memory and multi-agent systems. Other notable frameworks include Lindy, Mastra, and Haystack Agents. 

Links: 
https://www.lindy.ai/blog/best-ai-agent-frameworks 
https://www.instaclustr.com/education/agentic-ai/agentic-ai-frameworks-top-10-options-in-2026/ 
https://www.reddit.com/r/LangChain/comments/1rnc2u9/comprehensive_comparison_of_every_ai_agent/ 
https://www.a

### As always, take a look at the trace

https://platform.openai.com/traces


http://192.168.0.111:30606/projects

### We will now use Structured Outputs, and include a description of the fields

In [86]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 5

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

# https://console.groq.com/docs/structured-outputs#supported-models
planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        openai_client=groq_client,
    ),
    output_type=WebSearchPlan,
)

In [87]:

message = "Latest AI Agent frameworks in 2026"

with trace("Deep Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='This search will provide the most recent and relevant information on AI agent frameworks, as it directly matches the query.', query='latest AI agent frameworks 2026'), WebSearchItem(reason="This search term is similar to the first one but uses 'development' instead of 'latest', which might provide information on frameworks currently being developed or popular in 2026.", query='AI agent development frameworks 2026'), WebSearchItem(reason='This search focuses on autonomous AI agents, which could be a subset of AI agent frameworks, providing specific information on frameworks designed for autonomous agents.', query='autonomous AI agent frameworks'), WebSearchItem(reason="This search uses 'software agent' instead of just 'agent', which might yield results that include frameworks for agents in software applications, potentially highlighting frameworks used in various industries.", query='AI software agent frameworks'), WebSearchItem(reason="This search term u

In [88]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    msg = MIMEMultipart("alternative")
    msg["From"] = YAHOO_EMAIL
    msg["To"] = YAHOO_EMAIL
    msg["Subject"] = subject
    
    # Attach plain text fallback first, then HTML
    plain_text = "Please view this email in an HTML-compatible email client."
    msg.attach(MIMEText(plain_text, "plain"))
    msg.attach(MIMEText(html_body, "html"))
    
    with smtplib.SMTP_SSL("smtp.mail.yahoo.com", 465) as server:
        server.login(YAHOO_EMAIL, YAHOO_APP_PASSWORD)
        server.sendmail(YAHOO_EMAIL, YAHOO_EMAIL, msg.as_string())
    
    return {"status": "success"}

In [89]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f2c9c7a9f80>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [90]:
INSTRUCTIONS="""You are an email formatter and sender.

YOUR TASK (complete in exactly 2 steps):
1. Create ONE professional subject line based on the report content
2. Format the report as clean HTML and call send_email tool ONCE

CRITICAL RULES:
- Create EXACTLY ONE subject line do NOT create multiple options
- Call send_email EXACTLY ONE TIME with (html_content, subject)
- After calling send_email, output "TASK_COMPLETE" and STOP immediately
- DO NOT call send_email again
- DO NOT ask for confirmation or suggest alternatives
- Include ALL report content never trim or summarize

Subject Line Guidelines:
- Keep under 60 characters
- Make it descriptive but concise

If you follow these rules, you will send exactly ONE email with ONE subject line and then finish.
"""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        openai_client=groq_client,
    ),
)



In [91]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        openai_client=groq_client,
    ),
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [92]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [93]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [94]:
query ="Latest AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")




Starting research...
Planning searches...
Will perform 5 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


In [65]:
# Use the raw groq client instead of openai wrapper to inspect headers
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
response = client.chat.completions.with_raw_response.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[{"role": "user", "content": "hi"}],
)

print("Requests remaining:", response.headers.get("x-ratelimit-remaining-requests"))
print("Tokens remaining:  ", response.headers.get("x-ratelimit-remaining-tokens"))
print("Tokens reset at:   ", response.headers.get("x-ratelimit-reset-tokens"))

Requests remaining: 961
Tokens remaining:   29662
Tokens reset at:    675ms


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>